# TaaS Core Studio — Colab CPU / T4 profile

This notebook runs the experimental FastAPI discovery workspace from a pinned TaaS repository revision. Colab is **ephemeral compute**, not persistent storage or a production host. The current TaaS core is CPU-oriented; detecting a T4 does not imply that discovery is GPU-accelerated.

**Security boundary:** the API binds to `127.0.0.1`. Do not expose it through an unauthenticated public tunnel. If remote browser access is required, place it behind an authenticated tunnel or identity-aware proxy. The API bearer token protects discovery requests; the static UI and health endpoints are not independently authenticated, so use a protected tunnel for the whole service.

The notebook deliberately does not create a public tunnel or print secrets.

In [ ]:
import os, sys, subprocess, pathlib, json, time

REPO_URL = "https://github.com/msellamiTN/taas-provenance-discovery.git"
# Reproducible baseline for this notebook; update only after reviewing a newer PR revision.
TAAS_COMMIT = "fe94a55bb990cd9208249ae61567190e877d8e01"
WORKDIR = pathlib.Path("/content/taas-provenance-discovery")

if not WORKDIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(WORKDIR)], check=True)
subprocess.run(["git", "fetch", "--all"], cwd=WORKDIR, check=True)
subprocess.run(["git", "checkout", "--detach", TAAS_COMMIT], cwd=WORKDIR, check=True)
print("Checked out TaaS revision:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=WORKDIR, text=True).strip())

In [ ]:
# Detect available accelerator. This is diagnostic only; current TaaS discovery is CPU-oriented.
gpu = subprocess.run(["bash", "-lc", "nvidia-smi"], capture_output=True, text=True)
if gpu.returncode == 0:
    print(gpu.stdout.splitlines()[0])
else:
    print("No NVIDIA GPU detected; continuing on CPU.")

subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[studio]"], cwd=WORKDIR, check=True)

## Configure secrets and model profile

In Colab, add `TAAS_STUDIO_API_TOKEN` under **Secrets** and grant this notebook access. Upload a reviewed TaaS `ModelProfile` JSON file in the next cell. Do not print either the token or profile contents. Keep provider API keys in Colab Secrets or an external secret manager; do not put them in notebook cells or experiment manifests.

In [ ]:
from google.colab import userdata, files

api_token = userdata.get("TAAS_STUDIO_API_TOKEN")
if not api_token:
    raise RuntimeError("Create the TAAS_STUDIO_API_TOKEN Colab Secret and grant notebook access.")
os.environ["TAAS_STUDIO_API_TOKEN"] = api_token

uploaded = files.upload()
json_files = [name for name in uploaded if name.lower().endswith(".json")]
if len(json_files) != 1:
    raise RuntimeError("Upload exactly one reviewed ModelProfile JSON file.")
profile_path = pathlib.Path("/content") / json_files[0]
with profile_path.open("r", encoding="utf-8") as f:
    profile_data = json.load(f)
if not isinstance(profile_data, dict):
    raise ValueError("ModelProfile JSON must contain a JSON object.")
os.environ["TAAS_MODEL_PROFILE"] = str(profile_path.resolve())
print("API token loaded; model profile file configured (contents hidden).")

In [ ]:
# Start the API on loopback only. The process and files disappear when the Colab runtime ends.
server_log = open("/content/taas-studio-uvicorn.log", "w")
server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "taas.studio.api:app", "--host", "127.0.0.1", "--port", "8000"],
    cwd=WORKDIR, stdout=server_log, stderr=subprocess.STDOUT
)
time.sleep(3)
if server.poll() is not None:
    server_log.flush()
    raise RuntimeError("Studio API failed to start. Inspect /content/taas-studio-uvicorn.log.")
print("Studio API process started on loopback port 8000.")

In [ ]:
import urllib.request

for endpoint in ("health", "ready"):
    with urllib.request.urlopen(f"http://127.0.0.1:8000/{endpoint}", timeout=10) as response:
        print(endpoint, response.status, response.read().decode("utf-8"))

print("For local notebook use, open the loopback URL through a supported Colab port-forwarding mechanism.")
print("For remote sharing, configure an authenticated tunnel / identity-aware proxy separately; do not use an unauthenticated quick tunnel.")

## Operational limits

- This is a pinned, experimental discovery UI; no MCP tool invocation endpoint is included.
- The notebook does not configure a public tunnel. Protect the full UI and API with an authenticated access layer before remote sharing.
- Colab sessions can terminate without warning. Export manifests, traces and artifacts to durable external storage before relying on them.
- Do not upload production secrets, personal data, or unreviewed tool metadata.
- A detected T4 is not evidence of GPU acceleration. Measure any future CUDA-backed component independently.
- Stop the server when finished with `server.terminate()`.